In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train = pd.read_csv("preprocessed_train_final.csv")
test = pd.read_csv("preprocessed_test_final.csv")


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\3696746844.py:6: DtypeWarning: Columns (0: product_id) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv("preprocessed_test_final.csv")


In [6]:
train_df = train.groupby('category_fixed').sample(n=1).reset_index(drop=True)
test_df = test.groupby('category').sample(n=1).reset_index(drop=True)
del train
del test
import gc
gc.collect()

20

In [7]:
import json
category_mapping = json.load(open('category_map.json', 'r', encoding='utf-8'))


In [8]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, WeightedRandomSampler

In [9]:
TEXT_COL = "text_for_model"  # <-- change if needed
LEAF_TRAIN_COL = "category_fixed" if "category_fixed" in train_df.columns else "category"
LEAF_TEST_COL  = "category_fixed" if "category_fixed" in test_df.columns else "category"

# per-row weights (optional)
if "sample_weight" not in train_df.columns:
    train_df["sample_weight"] = 1.0
if "sample_weight" not in test_df.columns:
    test_df["sample_weight"] = 1.0

# derive main labels from leaf (your mapping)
train_df["main_true"] = train_df[LEAF_TRAIN_COL].astype(str).map(category_mapping).fillna("Diğer")
test_df["main_true"]  = test_df[LEAF_TEST_COL].astype(str).map(category_mapping).fillna("Diğer")

print("Train:", train_df.shape, "| Test:", test_df.shape)
train_df[[TEXT_COL, LEAF_TRAIN_COL, "main_true", "sample_weight"]].head()


Train: (1140, 37) | Test: (1140, 14)


,text_for_model,category_fixed,main_true,sample_weight
0,5 dbi 2 4 5 ghz wireless wifi metal yapıştırma...,ADSL Modemler,Elektronik,1.0
1,kiremit rengi prizma abajur ampul,Abajur,Ev & Yaşam,1.0
2,kadın mor komple gül desenli abiye elbise dd2408,Abiye & Mezuniyet Elbisesi,Giyim,1.0
3,yeşil suet kadin stiletto,Abiye Ayakkabı,Ayakkabı,1.0
4,kurşun simli kadın abiye çanta c06960200004,Abiye Çanta,Aksesuar,1.0


In [10]:
def topk_idx_from_logits(logits: np.ndarray, k: int) -> np.ndarray:
    idx = np.argpartition(-logits, kth=k-1, axis=1)[:, :k]
    row = np.arange(logits.shape[0])[:, None]
    idx_sorted = idx[row, np.argsort(-logits[row, idx], axis=1)]
    return idx_sorted

def topk_accuracy(y_true: np.ndarray, topk_idx: np.ndarray) -> float:
    return float(np.mean([y_true[i] in topk_idx[i] for i in range(len(y_true))]))

def metrics_from_logits(logits: np.ndarray, y_true: np.ndarray) -> dict:
    pred = np.argmax(logits, axis=1)
    top3 = topk_idx_from_logits(logits, 3)
    top5 = topk_idx_from_logits(logits, 5)
    return {
        "acc": float(accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, pred, average="weighted", zero_division=0)),
        "top3_acc": float(topk_accuracy(y_true, top3)),
        "top5_acc": float(topk_accuracy(y_true, top5)),
    }


In [11]:
def build_label_maps(series: pd.Series):
    le = LabelEncoder()
    le.fit(series.astype(str).values)
    id2label = {i: lab for i, lab in enumerate(le.classes_)}
    label2id = {lab: i for i, lab in id2label.items()}
    return le, label2id, id2label

def make_hf_dataset(df: pd.DataFrame, text_col: str, label_col: str, label2id: dict, tokenizer, max_len: int):
    tmp = df[[text_col, label_col, "sample_weight"]].copy()
    tmp[text_col] = tmp[text_col].fillna("").astype(str)
    tmp["label"] = tmp[label_col].astype(str).map(label2id)

    # drop rows with unseen labels (shouldn't happen for your test, but safe)
    tmp = tmp.dropna(subset=["label"]).copy()
    tmp["label"] = tmp["label"].astype(int)

    ds = Dataset.from_pandas(tmp[[text_col, "label", "sample_weight"]], preserve_index=False)

    def tok(batch):
        return tokenizer(batch[text_col], truncation=True, padding="max_length", max_length=max_len)

    ds = ds.map(tok, batched=True, remove_columns=[text_col])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch")
    return ds

def compute_class_and_sampler_weights(labels_np: np.ndarray, sample_weight_np: np.ndarray, alpha: float = 0.5):
    """
    class_weight[label] = (1/freq)^alpha normalized to mean=1
    sampler_weight[i] = sample_weight[i] * class_weight[labels[i]]
    """
    labels_np = labels_np.astype(int)
    n_classes = int(labels_np.max()) + 1

    freq = np.bincount(labels_np, minlength=n_classes).astype(np.float64)
    freq = np.maximum(freq, 1.0)

    class_w = (1.0 / freq) ** alpha
    class_w = class_w / (class_w.mean() + 1e-12)  # normalize

    sampler_w = sample_weight_np.astype(np.float64) * class_w[labels_np]
    sampler_w = sampler_w / (sampler_w.mean() + 1e-12)

    return class_w.astype(np.float32), sampler_w.astype(np.float64)


In [12]:
class BalancedWeightedTrainer(Trainer):
    def __init__(self, *args, sampler_weights=None, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._sampler_weights = sampler_weights  # numpy float64 length N
        self._class_weights = class_weights      # numpy float32 length C

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")

        data_collator = self.data_collator
        if data_collator is None:
            data_collator = DataCollatorWithPadding(self.tokenizer)

        # WeightedRandomSampler (balanced sampling)
        if self._sampler_weights is not None:
            w = torch.as_tensor(self._sampler_weights, dtype=torch.double)
            sampler = WeightedRandomSampler(
                weights=w,
                num_samples=len(w),      # keep epoch length ~dataset size
                replacement=True
            )
            return DataLoader(
                self.train_dataset,
                batch_size=self.args.train_batch_size,
                sampler=sampler,
                collate_fn=data_collator,
                num_workers=0,
                pin_memory=True
            )

        return super().get_train_dataloader()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        sample_w = inputs.get("sample_weight", None)

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            token_type_ids=inputs.get("token_type_ids", None),
            labels=None
        )
        logits = outputs.logits

        # per-example CE
        loss_vec = torch.nn.functional.cross_entropy(logits, labels, reduction="none")

        # class weights
        if self._class_weights is not None:
            cw = torch.tensor(self._class_weights, device=loss_vec.device, dtype=loss_vec.dtype)
            loss_vec = loss_vec * cw[labels]

        # sample weights
        if sample_w is not None:
            loss_vec = loss_vec * sample_w.to(loss_vec.dtype)

        loss = loss_vec.mean()
        return (loss, outputs) if return_outputs else loss


In [14]:
torch.backends.cuda.matmul.allow_tf32 = True

BASE_MODEL = "Trendyol/tyroberta"   # TR+EN mixed titles
MAX_LEN_MAIN = 96

tokenizer_main = AutoTokenizer.from_pretrained(BASE_MODEL)

# label space
main_le, main_label2id, main_id2label = build_label_maps(train_df["main_true"])

# datasets
ds_main_train = make_hf_dataset(train_df, TEXT_COL, "main_true", main_label2id, tokenizer_main, MAX_LEN_MAIN)
ds_main_test  = make_hf_dataset(test_df,  TEXT_COL, "main_true", main_label2id, tokenizer_main, MAX_LEN_MAIN)

# compute class weights + sampler weights for balanced sampling
y_main_train = np.array(ds_main_train["labels"])
sw_main_train = np.array(ds_main_train["sample_weight"], dtype=np.float32)
main_class_w, main_sampler_w = compute_class_and_sampler_weights(y_main_train, sw_main_train, alpha=0.5)

model_main = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(main_label2id),
    id2label=main_id2label,
    label2id=main_label2id
)

args_main = TrainingArguments(
    output_dir="bert_stage1_main_balanced",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=1,
    fp16=True,
    weight_decay=0.01,
    warmup_ratio=0.04,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
)

trainer_main = BalancedWeightedTrainer(
    model=model_main,
    args=args_main,
    train_dataset=ds_main_train,
    eval_dataset=ds_main_test,
    tokenizer=tokenizer_main,
    data_collator=DataCollatorWithPadding(tokenizer_main),
    sampler_weights=main_sampler_w,
    class_weights=main_class_w
)

trainer_main.train()

pred_main = trainer_main.predict(ds_main_test)
main_metrics = metrics_from_logits(pred_main.predictions, pred_main.label_ids)
print("Stage-1 MAIN metrics:", main_metrics)


Map: 100%|██████████| 1140/1140 [00:00<00:00, 23984.52 examples/s]
C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\2486134485.py:16: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_main_train = np.array(ds_main_train["labels"])
C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\2486134485.py:17: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sw_main_train = np.array(ds_main_train["sample_weight"], dtype=np.float32)
Some weights of RobertaForSequenceClassification were not i

Epoch,Training Loss,Validation Loss
1,No log,1.386572


Stage-1 MAIN metrics: {'acc': 0.41842105263157897, 'macro_f1': 0.45142430963104535, 'weighted_f1': 0.47709298209999396, 'top3_acc': 0.7570175438596491, 'top5_acc': 0.8991228070175439}


In [15]:
def predict_top2_labels(trainer, ds, id2label: dict):
    out = trainer.predict(ds)
    logits = out.predictions
    top2 = topk_idx_from_logits(logits, 2)

    top1_ids = top2[:, 0]
    top2_ids = top2[:, 1]

    top1_lab = np.array([id2label[int(i)] for i in top1_ids], dtype=object)
    top2_lab = np.array([id2label[int(i)] for i in top2_ids], dtype=object)

    # softmax probs for optional confidence
    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    p1 = probs[np.arange(len(probs)), top1_ids].astype(np.float32)
    p2 = probs[np.arange(len(probs)), top2_ids].astype(np.float32)
    return top1_lab, top2_lab, p1, p2

train_main_top1, train_main_top2, train_p1, train_p2 = predict_top2_labels(trainer_main, ds_main_train, main_id2label)
test_main_top1,  test_main_top2,  test_p1,  test_p2  = predict_top2_labels(trainer_main, ds_main_test,  main_id2label)

train_df["main_pred1"] = train_main_top1
train_df["main_pred2"] = train_main_top2
train_df["main_p1"] = train_p1
train_df["main_p2"] = train_p2

test_df["main_pred1"] = test_main_top1
test_df["main_pred2"] = test_main_top2
test_df["main_p1"] = test_p1
test_df["main_p2"] = test_p2

train_df[["main_pred1","main_pred2","main_p1","main_p2"]].head()


,main_pred1,main_pred2,main_p1,main_p2
0,Elektronik,Oyuncak,0.224541,0.156106
1,Ev & Yaşam,Oyuncak,0.143201,0.133023
2,Giyim,Oyuncak,0.168037,0.139036
3,Ayakkabı,Oyuncak,0.173847,0.146609
4,Oyuncak,Ayakkabı,0.148099,0.137472


In [16]:
def add_main_tokens(df: pd.DataFrame, base_text_col: str) -> pd.Series:
    return (
        df[base_text_col].fillna("").astype(str)
        + " __MAIN1__ " + df["main_pred1"].astype(str)
        + " __MAIN2__ " + df["main_pred2"].astype(str)
    )

train_df["text_plus_main2"] = add_main_tokens(train_df, TEXT_COL)
test_df["text_plus_main2"]  = add_main_tokens(test_df,  TEXT_COL)

train_df[["text_plus_main2"]].head(2)


,text_plus_main2
0,5 dbi 2 4 5 ghz wireless wifi metal yapıştırma...
1,kiremit rengi prizma abajur ampul __MAIN1__ Ev...


In [17]:
MAX_LEN_LEAF = 96
tokenizer_leaf = AutoTokenizer.from_pretrained(BASE_MODEL)

leaf_le, leaf_label2id, leaf_id2label = build_label_maps(train_df[LEAF_TRAIN_COL])

# Baseline datasets (text only)
ds_leaf_train_base = make_hf_dataset(train_df, TEXT_COL, LEAF_TRAIN_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF)
ds_leaf_test_base  = make_hf_dataset(test_df,  TEXT_COL, LEAF_TEST_COL,  leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF)

# Main-augmented datasets
ds_leaf_train_main = make_hf_dataset(train_df, "text_plus_main2", LEAF_TRAIN_COL, leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF)
ds_leaf_test_main  = make_hf_dataset(test_df,  "text_plus_main2", LEAF_TEST_COL,  leaf_label2id, tokenizer_leaf, MAX_LEN_LEAF)

# class + sampler weights for leaf training (balanced sampling)
y_leaf_train = np.array(ds_leaf_train_main["labels"])
sw_leaf_train = np.array(ds_leaf_train_main["sample_weight"], dtype=np.float32)
leaf_class_w, leaf_sampler_w = compute_class_and_sampler_weights(y_leaf_train, sw_leaf_train, alpha=0.5)

print("Leaf classes:", len(leaf_label2id))


Map: 100%|██████████| 1140/1140 [00:00<00:00, 17865.11 examples/s]

Leaf classes: 1140



C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\4256140722.py:15: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_leaf_train = np.array(ds_leaf_train_main["labels"])
C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\4256140722.py:16: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sw_leaf_train = np.array(ds_leaf_train_main["sample_weight"], dtype=np.float32)


In [18]:
model_leaf_main = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(leaf_label2id),
    id2label=leaf_id2label,
    label2id=leaf_label2id
)

args_leaf_main = TrainingArguments(
    output_dir="bert_stage2_leaf_with_main2_balanced",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    fp16=True,
    weight_decay=0.01,
    warmup_ratio=0.04,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=200,
    report_to="none",
)

trainer_leaf_main = BalancedWeightedTrainer(
    model=model_leaf_main,
    args=args_leaf_main,
    train_dataset=ds_leaf_train_main,
    eval_dataset=ds_leaf_test_main,
    tokenizer=tokenizer_leaf,
    data_collator=DataCollatorWithPadding(tokenizer_leaf),
    sampler_weights=leaf_sampler_w,
    class_weights=leaf_class_w
)

trainer_leaf_main.train()
pred_leaf_main = trainer_leaf_main.predict(ds_leaf_test_main)
metrics_leaf_main = metrics_from_logits(pred_leaf_main.predictions, pred_leaf_main.label_ids)
print("Stage-2 LEAF + MAIN(top2) metrics:", metrics_leaf_main)

# store predicted labels column
test_df["pred_leaf_bert_main2"] = [leaf_id2label[int(i)] for i in np.argmax(pred_leaf_main.predictions, axis=1)]


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Trendyol/tyroberta and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\ilker\AppData\Local\Temp\ipykernel_10716\891061367.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `BalancedWeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,7.055097


Stage-2 LEAF + MAIN(top2) metrics: {'acc': 0.002631578947368421, 'macro_f1': 0.00019512895919954028, 'weighted_f1': 0.00019512895919954028, 'top3_acc': 0.0035087719298245615, 'top5_acc': 0.0061403508771929825}


In [20]:
LEAF_TRAIN_COL = "category_fixed" if "category_fixed" in train_df.columns else "category"
LEAF_TEST_COL  = "category_fixed" if "category_fixed" in test_df.columns else "category"

# Build train frequency table
train_freq = train_df[LEAF_TRAIN_COL].astype(str).value_counts()
freq_tbl = train_freq.rename("train_cnt").reset_index().rename(columns={"index": "category_fixed"})
freq_tbl["cum_cnt"] = freq_tbl["train_cnt"].cumsum()
freq_tbl["cum_share"] = freq_tbl["cum_cnt"] / freq_tbl["train_cnt"].sum()

def to_hbt(cum_share):
    if cum_share <= 0.80:
        return "head"
    elif cum_share <= 0.95:
        return "body"
    else:
        return "tail"

freq_tbl["hbt"] = freq_tbl["cum_share"].map(to_hbt)

hbt_map = dict(zip(freq_tbl["category_fixed"], freq_tbl["hbt"]))

# Quick summary
summary = (freq_tbl.groupby("hbt")
           .agg(n_labels=("category_fixed","count"), train_rows=("train_cnt","sum"))
           .assign(train_pct=lambda x: (x["train_rows"]/x["train_rows"].sum()*100).round(2))
          )
summary


,n_labels,train_rows,train_pct
hbt,,,
body,171,171,15.0
head,912,912,80.0
tail,57,57,5.0


In [22]:
from sklearn.metrics import accuracy_score, f1_score

BERT_PRED_COL = "pred_leaf_bert_main2"
assert BERT_PRED_COL in test_df.columns, f"Missing {BERT_PRED_COL} in test_df"

eval_df = test_df.copy()
eval_df["y_true"] = eval_df[LEAF_TEST_COL].astype(str)
eval_df["y_pred"] = eval_df[BERT_PRED_COL].astype(str)

# Assign each row to head/body/tail based on TRUE label's train frequency
eval_df["hbt"] = eval_df["y_true"].map(hbt_map).fillna("tail")

def metric_block(df):
    yt = df["y_true"].values
    yp = df["y_pred"].values
    return {
        "n_rows": int(len(df)),
        "n_labels": int(pd.Series(yt).nunique()),
        "acc": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(yt, yp, average="weighted", zero_division=0)),
    }

rows = []
for part in ["head", "body", "tail", "ALL"]:
    sub = eval_df if part == "ALL" else eval_df[eval_df["hbt"] == part]
    out = metric_block(sub)
    out["slice"] = part
    rows.append(out)

hbt_metrics = pd.DataFrame(rows)[["slice","n_rows","n_labels","acc","macro_f1","weighted_f1"]]
hbt_metrics[["acc","macro_f1","weighted_f1"]] = hbt_metrics[["acc","macro_f1","weighted_f1"]].round(4)
hbt_metrics


,slice,n_rows,n_labels,acc,macro_f1,weighted_f1
0,head,912,912,0.0033,0.0003,0.0003
1,body,171,171,0.0000,0.0000,0.0000
2,tail,57,57,0.0000,0.0000,0.0000
3,ALL,1140,1140,0.0026,0.0002,0.0002
